# NB28: Web Dashboard & REST API
## RELIC's Interface Layer

RELIC provides multiple user interfaces:
- **CLI**: Command-line interface (covered in NB04)
- **REST API**: Programmatic access via HTTP (FastAPI, OpenAPI schema)
- **Web interface**: Browser pages for health monitoring and benchmarks, with
  experimental data and configuration pages
- **Jupyter Notebooks**: Interactive exploration (this tutorial series)

**Key module**: `relic.interfaces`

---

## Architecture Anchor

This notebook is aligned with `docs\source\architecture_overview.md` and the relevant theory/API pages under `docs\source`.
All computations below are intended to use actual RELIC functors, the actual simulator or orchestration layer, and real validation pathways rather than notebook-defined pseudo-process equations.


In [1]:
# Use only the standard library so notebook execution stays self-contained on Windows.
import asyncio
if hasattr(asyncio, "WindowsSelectorEventLoopPolicy"):
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

# --- Environment Guard (prevent OpenMP duplicate lib crash) ---
import os
import sys
import warnings
from collections import Counter, defaultdict
from pathlib import Path
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["RELIC_API_KEY"] = "nb28_local_demo_key"
warnings.filterwarnings("ignore", message="The pynvml package is deprecated.*")

DEFAULT_PROJECT_ROOT = Path(os.environ.get("RELIC_PROJECT_ROOT", Path.cwd())).resolve()

def locate_project_root() -> Path:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents, DEFAULT_PROJECT_ROOT]
    for candidate in candidates:
        if (candidate / "relic").is_dir() and (candidate / "docs" / "source" / "architecture_overview.md").is_file():
            return candidate
    raise FileNotFoundError("Could not locate RELIC project root from the active notebook environment.")

PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import inspect

# RELIC Interface imports
from relic.interfaces import web, api, cli
from relic.interfaces.web.app import create_web_app, create_api_app

api_app = create_api_app(enable_cors=False, enable_docs=True, debug=False, enable_auth=False)
web_app = create_web_app(enable_experimental=False, enable_cors=False, debug=False)
api_openapi = api_app.openapi()


def describe_route(route):
    return {
        "kind": type(route).__name__,
        "path": getattr(route, "path", None),
        "methods": sorted(getattr(route, "methods", [])) if getattr(route, "methods", None) else [],
        "name": getattr(route, "name", None),
    }

api_routes = [describe_route(route) for route in api_app.routes]
web_routes = [describe_route(route) for route in web_app.routes]

print("[OK] Interface modules and app factories loaded")
print(f"  project_root: {PROJECT_ROOT}")
print(f"  API app title: {api_app.title}")
print(f"  Web app title: {web_app.title}")
print(f"  API route count: {len(api_routes)}")
print(f"  Web route count: {len(web_routes)}")
print(f"  OpenAPI path count: {len(api_openapi.get('paths', {}))}")


[OK] Interface modules and app factories loaded
  project_root: <RELIC_ROOT>
  API app title: RELIC REST API
  Web app title: RELIC Web Interface
  API route count: 42
  Web route count: 12
  OpenAPI path count: 36


## 1. Web Dashboard Architecture

`relic.interfaces.web.app.create_web_app` builds a FastAPI app that renders Jinja2
templates, serves static assets under `/static`, and mounts the REST API app under
`/api`. Its routes are:
- `/`: home page
- `/health`: system health dashboard
- `/compute`: computation benchmarking and monitoring page
- `/data` (data file inspection) and `/config` (configuration viewer and validator):
  rendered only with experimental features enabled (`enable_experimental=True`, or
  `RELIC_EXPERIMENTAL_WEB=1` when the argument is left `None`); otherwise they return
  a feature-disabled page (this notebook passes `enable_experimental=False`)
- `/status`: JSON health payload used by the pages

The pages do not run or steer simulations, tune parameters, or display conservation
audits; graph compilation and submission are REST API endpoints (`/graph/...`).


In [2]:
# === NB28: Web app module structure and real mounted routes ===
import importlib

web_mod = importlib.import_module("relic.interfaces.web")
web_dir = Path(web_mod.__file__).resolve().parent
web_files = sorted(path.name for path in web_dir.glob("*.py"))
web_mounts = [route for route in web_routes if route["kind"] == "Mount"]
web_pages = [route for route in web_routes if route["kind"] == "APIRoute"]

print("=== Web App Module ===")
print("  Web files:")
for name in web_files:
    print(f"    {name}")

print("\n=== Mounted Components ===")
for route in web_mounts:
    print(f"  mount path={route['path']} name={route['name']}")

print("\n=== Web Page Routes ===")
for route in web_pages:
    methods = "/".join(route["methods"])
    print(f"  {methods:8s} {route['path']:12s} -> {route['name']}")

print("\n[OK] Web interface is backed by a real FastAPI app with mounted static and API subapps")


=== Web App Module ===
  Web files:
    __init__.py
    app.py

=== Mounted Components ===
  mount path=/static name=static
  mount path=/api name=api

=== Web Page Routes ===
  GET      /            -> home_page
  GET      /health      -> health_page
  GET      /data        -> data_page
  GET      /compute     -> compute_page
  GET      /config      -> config_page
  GET      /status      -> get_status

[OK] Web interface is backed by a real FastAPI app with mounted static and API subapps


## 2. REST API Endpoints

The REST API provides programmatic access to RELIC functionality
for integration with external tools and workflows.


In [3]:
# Examine API module structure and OpenAPI endpoint inventory
api_dir = Path(inspect.getfile(api)).resolve().parent
api_files = sorted(path.name for path in api_dir.glob("*.py"))
api_groups = defaultdict(list)
for path in sorted(api_openapi.get("paths", {})):
    parts = [part for part in path.split("/") if part]
    group = parts[0] if parts else "root"
    api_groups[group].append(path)

print("=== REST API Module ===")
print("  API files:")
for name in api_files:
    print(f"    {name}")

print("\n=== OpenAPI Endpoint Groups ===")
for group_name in sorted(api_groups):
    print(f"  {group_name}: {len(api_groups[group_name])} paths")
    for sample_path in api_groups[group_name][:4]:
        print(f"    {sample_path}")

print("\n=== API Metadata ===")
print(f"  title: {api_openapi.get('info', {}).get('title')}")
print(f"  version: {api_openapi.get('info', {}).get('version')}")
print(f"  docs route enabled: {'/docs' in [route['path'] for route in api_routes]}")
print(f"  metrics route present: {'/health/metrics' in [route['path'] for route in api_routes]}")
print(f"  auth middleware configured: {bool(getattr(api_app.state, 'api_auth_config', {}))}")
print("[OK] REST API structure examined from the live FastAPI app")


=== REST API Module ===
  API files:
    __init__.py
    dependencies.py
    middleware.py
    schemas.py
    server.py

=== OpenAPI Endpoint Groups ===
  compute: 4 paths
    /compute/
    /compute/benchmark
    /compute/ddp-check
    /compute/healthcheck
  config: 3 paths
    /config/
    /config/effective
    /config/validate
  data: 5 paths
    /data/
    /data/convert
    /data/inspect
    /data/read
  graph: 5 paths
    /graph/artifacts/{run_id}/{artifact_id}
    /graph/compile
    /graph/history/{run_id}
    /graph/submit
  health: 9 paths
    /health/
    /health/benchmark
    /health/compute
    /health/config
  optimization: 5 paths
    /optimization/
    /optimization/assimilate
    /optimization/calibrate
    /optimization/sensitivity
  root: 1 paths
    /
  stochastic: 4 paths
    /stochastic/
    /stochastic/gillespie
    /stochastic/sde
    /stochastic/weather

=== API Metadata ===
  title: RELIC REST API
  version: 1.0.0
  docs route enabled: True
  metrics route presen

## 2.1 Live REST Endpoint Verification

Use `fastapi.testclient.TestClient` to query the real RELIC FastAPI app in-process. This verifies actual response payloads, Prometheus export content, and warning behavior without launching an external server. All request models use `extra="forbid"`:
unknown fields must return HTTP 422. The negative request below verifies the
real boundary without applying a configuration change.

The health response includes its actual normalized status in the message: `System health check completed: HEALTHY` or `...: DEGRADED`. HEALTHY returns HTTP 200 with body status `success`; DEGRADED returns HTTP 200 with body status `warning`. Other health states return HTTP 503/error. A successful HTTP health query is therefore not a claim that every subsystem is healthy. The expected HTTP 422 negative probe may also produce an API error log; its assertion verifies that deliberate rejection.


In [4]:
# References:
# - relic/interfaces/api/server.py.
# - fastapi.testclient.TestClient request lifecycle.
# Mathematical expression:
# - endpoint_success_rate = n_success / max(1, n_checked)
# - relic_metric_fraction = n_relic_metric_lines / max(1, n_metric_lines)
# Parameter meanings:
# - checked_paths: live HTTP paths verified against the real FastAPI app.
# - n_success: number of 2xx responses in the checked path set.
# - n_metric_lines: number of non-empty Prometheus payload lines.
# - n_relic_metric_lines: number of RELIC-specific metric lines in the exported payload.

from fastapi.testclient import TestClient

checked_paths = ["/", "/health/", "/health/metrics", "/openapi.json"]
api_endpoint_snapshots = {}
unknown_field_status = None
metrics_lines = []

with warnings.catch_warnings(record=True) as api_warning_records:
    warnings.simplefilter("always")
    warnings.filterwarnings(
        "ignore",
        message="Consolidated metadata is currently not part in the Zarr format 3 specification.*",
    )
    live_api_app = create_api_app(enable_cors=False, enable_docs=True, debug=False, enable_auth=False)
    with TestClient(live_api_app) as client:
        rejected = client.post('/config/validate', json={'config': {}, 'unknown_tutorial_field': True})
        unknown_field_status = rejected.status_code
        assert unknown_field_status == 422, rejected.text
        for path in checked_paths:
            response = client.get(path)
            content_type = response.headers.get("content-type", "")
            snapshot = {
                "status_code": response.status_code,
                "content_type": content_type,
            }
            if "application/json" in content_type:
                payload = response.json()
                if path == "/health/":
                    snapshot["overall_status"] = payload["data"]["overall_status"]
                    snapshot["body_status"] = payload["status"]
                snapshot["json_keys"] = sorted(payload.keys()) if isinstance(payload, dict) else []
                if isinstance(payload, dict) and isinstance(payload.get("data"), dict):
                    snapshot["data_keys"] = sorted(payload["data"].keys())
                else:
                    snapshot["data_keys"] = []
                snapshot["message"] = payload.get("message") if isinstance(payload, dict) else None
            else:
                metrics_lines = [line for line in response.text.splitlines() if line.strip()]
                snapshot["line_count"] = len(metrics_lines)
            api_endpoint_snapshots[path] = snapshot

relic_help_lines = [line for line in metrics_lines if line.startswith("# HELP relic_")]
relic_primary_help_lines = [
    line for line in relic_help_lines if not line.split()[2].endswith("_created")
]
relic_series_lines = [line for line in metrics_lines if line.startswith("relic_")]
relic_primary_metric_names = sorted(
    {line.split()[2] for line in relic_primary_help_lines}
)
relic_primary_series_lines = [
    line for line in relic_series_lines if "_created" not in line.split()[0]
]
relic_metric_sample_lines = relic_primary_help_lines[:2] + relic_primary_series_lines[:4]
api_warning_count_by_kind = Counter(type(record.message).__name__ for record in api_warning_records)
api_warning_examples = {
    warning_kind: sorted(
        {
            str(record.message)
            for record in api_warning_records
            if type(record.message).__name__ == warning_kind
        }
    )[:2]
    for warning_kind in sorted(api_warning_count_by_kind)
}
endpoint_success_count = sum(
    1 for snapshot in api_endpoint_snapshots.values() if 200 <= snapshot["status_code"] < 300
)
api_live_summary = {
    "checked_paths": checked_paths,
    "endpoint_status_codes": {
        path: api_endpoint_snapshots[path]["status_code"] for path in checked_paths
    },
    "endpoint_success_rate": endpoint_success_count / max(1, len(checked_paths)),
    "root_data_keys": api_endpoint_snapshots["/"]["data_keys"],
    "health_data_keys": api_endpoint_snapshots["/health/"]["data_keys"],
    "health_overall_status": api_endpoint_snapshots["/health/"]["overall_status"],
    "health_body_status": api_endpoint_snapshots["/health/"]["body_status"],
    "relic_metric_family_count": len(relic_primary_metric_names),
    "relic_metric_names": relic_primary_metric_names,
    "metrics_line_count": len(metrics_lines),
    "relic_metric_line_count": len(relic_primary_help_lines) + len(relic_primary_series_lines),
    "metrics_sample_lines": relic_metric_sample_lines,
    "api_warning_count": sum(api_warning_count_by_kind.values()),
    "unknown_request_field_status": unknown_field_status,
    "api_warning_count_by_kind": dict(api_warning_count_by_kind),
    "api_warning_examples": api_warning_examples,
}

assert api_live_summary["endpoint_success_rate"] == 1.0
assert api_live_summary["relic_metric_family_count"] >= 4
assert api_live_summary["metrics_line_count"] > 0
assert api_endpoint_snapshots["/"]["message"] == "RELIC REST API is running"
# Health route appends the actual status and keeps DEGRADED as HTTP 200/warning.
health_status = api_live_summary["health_overall_status"]
assert health_status in {"HEALTHY", "DEGRADED"}
assert api_live_summary["health_body_status"] == {"HEALTHY": "success", "DEGRADED": "warning"}[health_status]
assert api_endpoint_snapshots["/health/"]["message"] == f"System health check completed: {health_status}"

print("=== Live RELIC API Endpoint Verification ===")
for path in checked_paths:
    snapshot = api_endpoint_snapshots[path]
    print(
        f"  {path:15s} status={snapshot['status_code']} content_type={snapshot['content_type']}"
    )

print("\n=== Response Payload Checks ===")
print(f"  Root data keys: {api_live_summary['root_data_keys']}")
print(f"  Health data keys: {api_live_summary['health_data_keys']}")
print(f"  Actual health status: {api_live_summary['health_overall_status']} / {api_live_summary['health_body_status']}")
print(f"  Expected unknown-field rejection status: {api_live_summary['unknown_request_field_status']}")
print(f"  OpenAPI path count: {len(api_openapi.get('paths', {}))}")

print("\n=== Prometheus Export Checks ===")
print(f"  RELIC metric families: {api_live_summary['relic_metric_family_count']}")
print(f"  RELIC metric names: {api_live_summary['relic_metric_names']}")
print(f"  Metrics payload lines: {api_live_summary['metrics_line_count']}")
for sample_line in api_live_summary["metrics_sample_lines"]:
    print(f"    {sample_line}")

print("\n=== Captured Runtime Warnings ===")
print(f"  warning_count = {api_live_summary['api_warning_count']}")
print(f"  warning_count_by_kind = {api_live_summary['api_warning_count_by_kind']}")
for warning_kind, messages in api_live_summary["api_warning_examples"].items():
    for message in messages:
        print(f"    {warning_kind}: {message}")

api_live_summary

API Error - POST http://testserver/config/validate from testclient: 1 validation error:
  {'type': 'extra_forbidden', 'loc': ('body', 'unknown_tutorial_field'), 'msg': 'Extra inputs are not permitted', 'input': True}

  File "relic\interfaces\api\routes\config.py", line 73, in validate_config
    POST /config/validate


Configuration loaded with 1 warnings


=== Live RELIC API Endpoint Verification ===
  /               status=200 content_type=application/json
  /health/        status=200 content_type=application/json
  /health/metrics status=200 content_type=text/plain; version=0.0.4; charset=utf-8
  /openapi.json   status=200 content_type=application/json

=== Response Payload Checks ===
  Root data keys: ['capabilities', 'docs', 'endpoints', 'title', 'version']
  Health data keys: ['errors', 'overall_status', 'recommendations', 'subsystems', 'timestamp', 'warnings']
  Actual health status: DEGRADED / warning
  Expected unknown-field rejection status: 422
  OpenAPI path count: 36

=== Prometheus Export Checks ===
  RELIC metric families: 4
  RELIC metric names: ['relic_api_auth_failures_total', 'relic_api_exceptions_total', 'relic_api_request_duration_seconds', 'relic_api_requests_total']
  Metrics payload lines: 94
    # HELP relic_api_requests_total Total API requests.
    # HELP relic_api_request_duration_seconds API request latency i

{'checked_paths': ['/', '/health/', '/health/metrics', '/openapi.json'],
 'endpoint_status_codes': {'/': 200,
  '/health/': 200,
  '/health/metrics': 200,
  '/openapi.json': 200},
 'endpoint_success_rate': 1.0,
 'root_data_keys': ['capabilities', 'docs', 'endpoints', 'title', 'version'],
 'health_data_keys': ['errors',
  'overall_status',
  'recommendations',
  'subsystems',
  'timestamp',
  'warnings'],
 'health_overall_status': 'DEGRADED',
 'health_body_status': 'warning',
 'relic_metric_family_count': 4,
 'relic_metric_names': ['relic_api_auth_failures_total',
  'relic_api_exceptions_total',
  'relic_api_request_duration_seconds',
  'relic_api_requests_total'],
 'metrics_line_count': 94,
 'relic_metric_line_count': 59,
 'metrics_sample_lines': ['# HELP relic_api_requests_total Total API requests.',
  '# HELP relic_api_request_duration_seconds API request latency in seconds.',
  'relic_api_requests_total{method="POST",path="/config/validate",status="422"} 1.0',
  'relic_api_requests_

## 3. CLI Integration

The RELIC CLI (`python -m relic.interfaces.cli.main <command>`) provides the
subcommands `params`, `optimization`, `data`, `compute`, `stochastic`,
`orchestration validate|compile`, and `healthcheck`; layered simulation runs have
their own entry point, `python -m relic.interfaces.cli.run_layered` (see NB04).


In [5]:
# Examine CLI structure and its relationship to the interface layer
cli_dir = Path(inspect.getfile(cli)).resolve().parent
cli_files = sorted(path.name for path in cli_dir.glob("*.py"))
# The package declares its public entry points in __all__.
cli_function_names = list(cli.__all__)

print("=== CLI Module ===")
print("  CLI files:")
for name in cli_files:
    print(f"    {name}")

print("\n=== Public CLI Entry Points (relic.interfaces.cli.__all__) ===")
for name in sorted(cli_function_names):
    obj = getattr(cli, name)
    print(f"  {name}{inspect.signature(obj)}")

print("\n=== Interface Layer Roles ===")
print("  CLI: batch and scripted workflows")
print("  REST API: machine-to-machine access with OpenAPI schema")
print("  Web: browser-facing pages mounted on a FastAPI app")
print("[OK] CLI structure examined")


=== CLI Module ===
  CLI files:
    __init__.py
    compute.py
    data.py
    healthcheck.py
    main.py
    optimization.py
    params.py
    run_layered.py
    stochastic.py
    utils.py

=== Public CLI Entry Points (relic.interfaces.cli.__all__) ===
  cli_entry_point(*args: Any, **kwargs: Any) -> int
  main(*args: Any, **kwargs: Any) -> int

=== Interface Layer Roles ===
  CLI: batch and scripted workflows
  REST API: machine-to-machine access with OpenAPI schema
  Web: browser-facing pages mounted on a FastAPI app
[OK] CLI structure examined


## 4. Interface Architecture Overview

```
User
 |
 |-- CLI (relic.interfaces.cli; python -m relic.interfaces.cli.main <command>)
 |     |-- params, optimization, data, compute, stochastic
 |     |-- orchestration validate | compile
 |     |-- healthcheck
 |     |-- run_layered (separate entry: python -m relic.interfaces.cli.run_layered)
 |
 |-- REST API (relic.interfaces.api, create_api_app)
 |     |-- /health/... (incl. /health/metrics, Prometheus text)
 |     |-- /config/..., /data/..., /compute/...
 |     |-- /graph/compile, /graph/submit, /graph/history/..., /graph/artifacts/...
 |     |-- /optimization/..., /stochastic/...
 |
 |-- Web interface (relic.interfaces.web, create_web_app)
 |     |-- pages: /, /health, /compute; /data, /config (experimental)
 |     |-- /status (JSON), /static, /api (mounted REST API)
 |
 |-- Jupyter: this tutorial series (relic.interfaces.notebooks is an empty package)
```


In [6]:
print("=== RELIC Interface Summary ===")
print(f"  API title: {api_app.title}")
print(f"  Web title: {web_app.title}")
print(f"  API paths in OpenAPI: {len(api_openapi.get('paths', {}))}")
print(f"  Web mounted API subtree: {'/api' in [route['path'] for route in web_routes]}")
print(f"  Web static assets mounted: {'/static' in [route['path'] for route in web_routes]}")
print(f"  CLI files discovered: {len(cli_files)}")
print(f"  Live endpoint success rate: {api_live_summary['endpoint_success_rate']:.2f}")
print(f"  RELIC metric families observed: {api_live_summary['relic_metric_family_count']}")
print(f"  Runtime warning count: {api_live_summary['api_warning_count']}")
print("\n[OK] Interface tutorial complete")


=== RELIC Interface Summary ===
  API title: RELIC REST API
  Web title: RELIC Web Interface
  API paths in OpenAPI: 36
  Web mounted API subtree: True
  Web static assets mounted: True
  CLI files discovered: 10
  Live endpoint success rate: 1.00
  RELIC metric families observed: 4
  Runtime warning count: 0

[OK] Interface tutorial complete
